In [12]:
dataset_size = 1
max_length = 256
lam = 256
model_name = "meta-llama/Llama-3.2-3B"
device = "0"
n_samples = 100

In [13]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('../src')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [14]:
import torch
from utils import load_model_and_tokenizer, load_c4

experiment = "varying_length"

device = torch.device(f"cuda:{device}" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model, tokenizer = load_model_and_tokenizer(model_name, device)

data = load_c4(tokenizer, dataset_size)
data[0]

Using device: cuda:0


Loading checkpoint shards: 100%|██████████| 2/2 [00:02<00:00,  1.16s/it]


Model: meta-llama/Llama-3.2-3B
Loading dataset from cache...


{'input_ids': tensor([[128000,    791,  59796,   6406,   8925,   6787,    311,   7055,   7884,
          13658,    389,   7231,   8339,    400,   1419,   3610,    323,   1023,
          36580,    311,   1977,    264,  26097,  15679,    304,  29016,   4409,
             11,    719,   1193,   1306,  11011,  12483,    315,  18671,  13286,
          11062,    323,  28424,  49262,    369,    477,   2403,    279,   2447,
            627,    791,   4330,  44650,   4580]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1]])}

In [15]:
from utils import (
    load_default_wepa_watermarker,
    load_default_exp_watermarker,
    load_default_kgw_watermarker,
)

seed = 42
vocab_size = tokenizer.vocab_size

kgw_wat = load_default_kgw_watermarker(vocab_size, device, seed=seed)
exp_wat = load_default_exp_watermarker(vocab_size, device, seed=seed)
wepa_d1_wat = load_default_wepa_watermarker(vocab_size, device, degree=1, seed=seed)
wepa_d2_wat = load_default_wepa_watermarker(vocab_size, device, degree=2, seed=seed)
wepa_d2_b6_wat = load_default_wepa_watermarker(vocab_size, device, degree=2, bits=6, seed=seed)

In [16]:
import time
from tqdm import tqdm
from utils import get_wat_name


def evaluate_time(watermarker, unoptimized=False):
    tokens = torch.tensor(list(range(max_length)), device=device)
    if not unoptimized:
        # for JIT compilation
        for i in range(3):
            watermarker.p_value(tokens, n_samples=n_samples)

        tic = time.time()
        for i in tqdm(range(dataset_size)):
            watermarker.p_value(data[i]["input_ids"][0], n_samples=n_samples)
        toc = time.time()
    else:
        # for JIT compilation
        for i in range(3):
            watermarker.p_value_unoptimized(tokens, n_samples=n_samples)

        tic = time.time()
        for i in tqdm(range(dataset_size)):
            watermarker.p_value_unoptimized(data[i]["input_ids"][0], n_samples=n_samples)
        toc = time.time()

    print(
        f"Watermarker: {get_wat_name(watermarker)}"
        f"{' (Unoptimized)' if unoptimized else ''}, "
        f"Time taken: {toc - tic:.3} s"
    )
    return toc - tic


result_wepa_d1 = evaluate_time(wepa_d1_wat)
result_wepa_d2 = evaluate_time(wepa_d2_wat)
result_wepa_d2_b6 = evaluate_time(wepa_d2_b6_wat)
result_exp = evaluate_time(exp_wat)
result_exp_unoptimized = evaluate_time(exp_wat, unoptimized=True)
result_kgw = evaluate_time(kgw_wat)

100%|██████████| 1/1 [00:00<00:00, 63.07it/s]


Watermarker: wepa_d1, Time taken: 0.0169 s


100%|██████████| 1/1 [00:00<00:00, 62.97it/s]


Watermarker: wepa_d2, Time taken: 0.0167 s


100%|██████████| 1/1 [00:00<00:00, 48.58it/s]


...

Watermarker: exp, Time taken: 0.17 s


100%|██████████| 1/1 [00:12<00:00, 12.91s/it]


Watermarker: exp (Unoptimized), Time taken: 12.9 s


100%|██████████| 1/1 [00:00<00:00,  2.50it/s]

Watermarker: kgw, Time taken: 0.402 s
